# Baseline Poisson Model

Homogeneous Poisson Process: 
crime arrives at a constant rate λ everywhere, all the time. 
This is your "dumb" baseline that everything else should beat.

Basically, we dont put any X features, but the model tries to predict solely based on the y values.


In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import numpy as np
import pandas as pd
import torch

import pyro
import pyro.distributions as dist
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.optim import Adam, ClippedAdam

from config import cfg

ImportError: cannot import name 'cfg' from 'config' (/Users/helgamariamagnusdottir/Documents/dtu/03_model_based_ml/model-based/src/models/config.py)

In [ ]:
# Start by loading the aggregated dataframe
df = pd.read_csv('../../data/processed/pd_2018_present_agg.csv')
df.head()

,year,month,day,weekday,hour,district,incident_count
0,2018,1,1,0,0,Bayview,16
1,2018,1,1,0,0,Central,18
2,2018,1,1,0,0,Ingleside,6
3,2018,1,1,0,0,Mission,10
4,2018,1,1,0,0,Northern,21


In [ ]:
# We only need the y values (incident count)
y = df['incident_count'].values

# Split that into train and test
train_perc = cfg.train_size
split_point = int(train_perc*len(y))
perm = np.random.permutation(len(y))
ix_train = perm[:split_point]
ix_test = perm[split_point:]

y_train = y[ix_train]
y_test = y[ix_test]

print("num train: %d" % len(y_train))
print("num test: %d" % len(y_test))


In [ ]:
# Change into tensors
y_train_torch = torch.tensor(y_train, dtype=torch.float32)
y_test_np = y_test.astype(float)

In [ ]:
# Define the baseline model
def constant_poisson_model(y=None):
    n = len(y)
    log_rate = pyro.sample("log_rate", dist.Normal(0., 1.))
    rate = torch.exp(log_rate)
    
    with pyro.plate("data", n):
        pyro.sample("obs", dist.Poisson(rate), obs=y)

In [ ]:
# Define params
n_steps = 3000
lr = 0.005

In [ ]:
# Now we train the model
pyro.clear_param_store()

guide = AutoDiagonalNormal(constant_poisson_model)

optimizer = ClippedAdam({"lr": lr})
svi = SVI(constant_poisson_model, guide, optimizer, loss=Trace_ELBO())

for step in range(n_steps):
    loss = svi.step(y_train_torch)
    
    if step % 500 == 0:
        print(f"[{step}] ELBO loss: {loss:.2f}")

[0] ELBO loss: 674170.70
[500] ELBO loss: 513271.47
[1000] ELBO loss: 513213.67
[1500] ELBO loss: 513188.16
[2000] ELBO loss: 513187.68
[2500] ELBO loss: 513189.29


In [ ]:
num_samples = 1000
n_test = len(y_test_np)

predictive = Predictive(
    constant_poisson_model,
    guide=guide,
    num_samples=num_samples,
    return_sites=("log_rate",)
)

dummy_y = torch.zeros(n_test)
samples = predictive(dummy_y)

rate_samples = torch.exp(samples["log_rate"]).squeeze()

mean_rate = rate_samples.mean().item()
y_pred = np.full(n_test, mean_rate)

In [ ]:
# Define function to compute error
def compute_error(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / denominator

    return mae, rmse, r2

In [ ]:
mae, rmse, r2 = compute_error(y_test_np, y_pred)

print("\nBaseline Poisson model")
print(f"Estimated constant rate prediction: {y_pred[0]:.3f}")
print(f"CorrCoef: undefined for constant baseline")
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")



Baseline Poisson model
Estimated constant rate prediction: 2.294
CorrCoef: undefined for constant baseline
MAE: 1.255
RMSE: 1.705
R2: -0.000
